# Strength Scan Overview

## Configuration

In [ ]:
from pathlib import Path
import os

SCAN_ROOT = Path(
    os.environ.get(
        "WWGPT_STRENGTH_SCAN_ROOT",
        "/tmp/wwpgd_strength_scan",
    )
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from wwgpt.strength_scan_analysis import resolve_scan_root, analyze_strength_scan
try:
    scan_root = resolve_scan_root(SCAN_ROOT)
except FileNotFoundError:
    scan_root = resolve_scan_root(Path("tests/fixtures/strength_scan"))
analysis_dir = analyze_strength_scan(scan_root)
fig_dir = scan_root / "analysis" / "notebook_overview"
fig_dir.mkdir(parents=True, exist_ok=True)
print(scan_root)

## Scan manifest

## Run inventory

## Completion and stability audit

## Pairing audit

## Terminal validation-loss comparison

## Validation-loss trajectories

## Projection magnitude

## Throughput and runtime

## Strength-response summary

## Preliminary interpretation

In [ ]:
import json
manifest=json.loads((scan_root/"scan_manifest.json").read_text()); display(manifest)
runs=pd.read_csv(analysis_dir/"strength_scan_runs.csv"); terminal=pd.read_csv(analysis_dir/"strength_scan_terminal.csv"); proj=pd.read_csv(analysis_dir/"strength_scan_projection.csv"); summary=pd.read_csv(analysis_dir/"strength_scan_summary.csv")
display(runs); display(runs[runs.complete==True]); display(runs[(runs.stable==False) | (runs.complete==False)]); display(terminal); display(summary)
if summary.seed_count.max() <= 1: print("One-seed scan: confidence intervals are unavailable.")
for y,name in [("wwpgd_minus_adamw_final_loss","final_loss_diff"),("wwpgd_minus_adamw_minimum_loss","min_loss_diff")]:
    ax=terminal.sort_values("strength").plot(x="strength", y=y, marker="o", legend=False); ax.set_title(name); ax.figure.savefig(fig_dir/f"{name}.png"); plt.close(ax.figure)
for y,name in [("median_relative_frobenius_change","projection_norm"),("total_projection_runtime","projection_runtime")]:
    if len(proj):
        ax=proj.groupby("strength")[y].mean().plot(marker="o"); ax.set_title(name); ax.figure.savefig(fig_dir/f"{name}.png"); plt.close(ax.figure)
for y,name in [("total_immediate_weightwatcher_runtime","immediate_ww"),("tokens_per_second","tokens_per_second")]:
    ax=runs[runs.optimizer_family=="wwpgd"].groupby("strength")[y].mean().plot(marker="o"); ax.set_title(name); ax.figure.savefig(fig_dir/f"{name}.png"); plt.close(ax.figure)
ax=runs[runs.optimizer_family=="wwpgd"].groupby("strength")["stable"].mean().plot(kind="bar"); ax.figure.savefig(fig_dir/"stability.png"); plt.close(ax.figure)
# trajectories
for _,r in runs.iterrows():
    mpath=Path(r.run_dir)/"metrics.csv"
    if mpath.exists():
        m=pd.read_csv(mpath); plt.plot(m.tokens_processed, m.val_loss, label=f"{r.optimizer_family} {r.strength}")
plt.legend(); plt.savefig(fig_dir/"val_trajectories.png"); plt.close()
print("Observed trade-offs are shown above; no optimal strength is selected automatically.")